<div class="not-slide">

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana09/EGC5310-EstruturasDados-S09-99-Aula-Mestre.ipynb)

**S09 · Mestre · versão 1.0 · 05/10/2026**  
Execute as células em ordem. Os exemplos pequenos tornam visíveis as estratégias de combinação; o caso Olist será usado depois para observar cardinalidade em dados reais.

</div>


## EGC5310 {.title-slide-custom}

<img src="../assets/Logo_03.png" class="course-logo" alt="Logo Ciência de Dados UFSC">

<div class="course-name">ESTRUTURAS DE DADOS</div>
<div class="week-label">SEMANA 09</div>
<div class="subtitle-custom">Merge/Join: combinando estruturas de dados</div>


## De onde viemos?

```text
Busca       → como encontrar?
Hashing     → como encontrar por uma chave?
Ordenação   → como organizar para operar melhor?
                         ↓
                    SEMANA 09
                         ↓
             Como combinar coleções?
```

**Hoje, estruturas e algoritmos anteriores reaparecem dentro de uma operação comum em Ciência de Dados.**

::: {.notes}
Abrir recuperando ideias, não APIs. A S08 foi integralmente apresentada. Perguntar onde hashing e ordenação poderiam ajudar se duas coleções compartilhassem uma chave.
:::


## O problema da semana {.question-slide}

Temos informações sobre estudantes em uma estrutura e notas em outra.

**Como produzir `matrícula + nome + nota`?**

Antes do código: descreva uma estratégia em linguagem natural.


In [ ]:
estudantes = [
    {"matricula": 101, "nome": "Ana"},
    {"matricula": 102, "nome": "Bruno"},
    {"matricula": 103, "nome": "Carla"},
    {"matricula": 104, "nome": "Diego"},
]

notas = [
    {"matricula": 103, "nota": 8.5},
    {"matricula": 101, "nota": 9.0},
    {"matricula": 104, "nota": 7.5},
]

print("Estudantes:", estudantes)
print("Notas:", notas)


## O que permite combinar?

| estudantes |  | notas |
|---|---|---|
| 101 · Ana | ↔ | 101 · 9.0 |
| 102 · Bruno | ? | — |
| 103 · Carla | ↔ | 103 · 8.5 |
| 104 · Diego | ↔ | 104 · 7.5 |

A **matrícula** aparece nas duas estruturas.

::: {.fragment}
Ela pode funcionar como **chave de correspondência**.
:::


## AGORA É COM VOCÊ {.question-slide}

**Notebook Estudante · Atividade 1 — Como combinar duas coleções?**

Sem usar `pandas`, descreva uma estratégia para obter nome e nota a partir das duas listas. Depois estime: com 4 estudantes e 3 registros de notas, quantas comparações podem ser necessárias no pior caso?

[**Abrir Notebook Estudante no Colab**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana09/EGC5310-EstruturasDados-S09-04-Estudante.ipynb)

Registre a previsão antes de executar qualquer solução.


## A solução mais literal

Para cada estudante:

1. percorra as notas;
2. compare as matrículas;
3. quando encontrar correspondência, produza um novo registro.

**A descrição já sugere dois percursos aninhados.**


## Percorrendo as duas estruturas {.smaller}

Os comentários destacam **por que** cada passo existe. Os `print`s tornam visível **o que o algoritmo faz**.

::: {.notes}
Não ler os comentários como texto. Usá-los para apontar responsabilidade das partes do código. Perguntar por que o break é seguro neste exemplo e quando deixaria de ser.
:::


In [ ]:
def combinar_nested_loop(estudantes, notas, mostrar_passos=False):
    resultado = []
    comparacoes = 0

    # Cada estudante precisa procurar sua matrícula na coleção de notas.
    for estudante in estudantes:
        encontrou = False

        # Sem uma estrutura auxiliar, a procura começa percorrendo as notas.
        for registro_nota in notas:
            comparacoes += 1

            if mostrar_passos:
                print(
                    f'{estudante["matricula"]} × '
                    f'{registro_nota["matricula"]}'
                )

            if estudante["matricula"] == registro_nota["matricula"]:
                # A correspondência permite construir um registro combinado.
                resultado.append({
                    "matricula": estudante["matricula"],
                    "nome": estudante["nome"],
                    "nota": registro_nota["nota"],
                })
                encontrou = True

                # Neste exemplo, cada matrícula ocorre no máximo uma vez em notas.
                break

        if mostrar_passos and not encontrou:
            print(f'  matrícula {estudante["matricula"]}: sem nota')

    return resultado, comparacoes


In [ ]:
resultado_nested, comparacoes_nested = combinar_nested_loop(
    estudantes,
    notas,
    mostrar_passos=True
)

print("\nResultado:", resultado_nested)
print("Comparações realizadas:", comparacoes_nested)


## O que os prints revelam?

```text
101 × 103
101 × 101  ✓

102 × 103
102 × 101
102 × 104   → não encontrou

103 × 103  ✓
...
```

O algoritmo **não sabe onde está a matrícula** antes de procurar.

> Encontrar cedo reduz o trabalho desta execução, mas não altera o problema estrutural.


## Quantas comparações?

Se nenhuma correspondência permitir interromper cedo:

$$n \times m$$

Para `n` registros de um lado e `m` do outro:

$$O(nm)$$

Se os tamanhos forem semelhantes:

$$O(n^2)$$


## Isso tem nome

### Nested-loop join

```text
para cada registro de A
    percorra B
        compare as chaves
```

É simples, correto em muitos cenários e pode ser suficiente para coleções pequenas.

**O problema aparece quando os dados crescem.**


In [ ]:
for n in [10, 100, 1_000, 100_000]:
    comparacoes = n * n
    print(f"{n:>7,} × {n:>7,} → {comparacoes:>15,} comparações potenciais")


## Funciona. Então qual é o problema? {.question-slide}

| A | B | Comparações potenciais |
|---:|---:|---:|
| 10 | 10 | 100 |
| 100 | 100 | 10.000 |
| 1.000 | 1.000 | 1.000.000 |
| 100.000 | 100.000 | 10.000.000.000 |

**Já conhecemos alguma representação que permita procurar uma matrícula sem percorrer toda a coleção?**


## Recuperando hashing

Se conhecemos a matrícula `103`, queremos algo próximo de:

```python
nota = indice_notas[103]
```

A pergunta muda:

> Vale a pena gastar trabalho antes para construir uma estrutura que torne as consultas seguintes mais baratas?


## Mudando a representação {.smaller}

Os dados continuam sendo os mesmos. O que muda é a organização usada para acessá-los.


In [ ]:
indice_notas = {}

for registro in notas:
    matricula = registro["matricula"]

    # A matrícula vira a chave do dicionário.
    # Assim, não precisamos procurar a nota percorrendo uma lista.
    indice_notas[matricula] = registro["nota"]

    print("Após inserir", matricula, "→", indice_notas)


## O dado mudou?

**Não.**

Mudou a **representação**:

```text
lista de registros
      ↓
tabela hash organizada por matrícula
      ↓
matrícula → nota
```

E a representação altera as operações que conseguimos realizar eficientemente.


## Combinando com o índice {.smaller}


In [ ]:
def combinar_hash(estudantes, notas):
    indice_notas = {}

    # Fase 1: pagamos o custo de organizar as notas por matrícula.
    for registro in notas:
        indice_notas[registro["matricula"]] = registro["nota"]

    resultado = []

    # Fase 2: cada estudante consulta diretamente a chave no índice.
    for estudante in estudantes:
        matricula = estudante["matricula"]

        if matricula in indice_notas:
            resultado.append({
                "matricula": matricula,
                "nome": estudante["nome"],
                "nota": indice_notas[matricula],
            })

    return resultado, indice_notas


In [ ]:
resultado_hash, indice = combinar_hash(estudantes, notas)

print("Índice:", indice)
print("Resultado:", resultado_hash)


## Onde foi parar o segundo `for`? {.question-slide}

Temos duas fases:

1. construir o índice: $O(m)$;
2. consultar uma chave para cada estudante: aproximadamente $O(n)$ em média.

Logo:

$$O(n+m)$$

para o comportamento médio esperado da tabela hash.

::: {.notes}
Reforçar: O(1) médio da consulta não significa uma operação física única. O ponto é que o trabalho de consulta não cresce proporcionalmente ao tamanho da coleção nas condições usuais da tabela hash.
:::


## Hash join

```text
coleção de notas
      ↓
construir hash
      ↓
matrícula → nota
      ↓
percorrer estudantes
      ↓
consultar matrícula
```

**Pagamos pela construção de uma estrutura auxiliar para reduzir o custo das buscas.**


## AGORA É COM VOCÊ {.question-slide}

**Notebook Estudante · Atividade 2 — O custo mudou onde?**

Compare nested-loop e hash join. Identifique:

- trabalho feito antes das consultas;
- trabalho repetido para cada estudante;
- estrutura auxiliar criada;
- hipótese usada ao falar em consulta `O(1)` média.

[**Abrir Notebook Estudante no Colab**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana09/EGC5310-EstruturasDados-S09-04-Estudante.ipynb)


## E a ordenação da semana passada? {.question-slide}

E se as duas coleções estiverem ordenadas pela mesma chave?

```text
ESTUDANTES          NOTAS
101 Ana             101 9.0
102 Bruno           103 8.5
103 Carla           104 7.5
104 Diego
```

**Precisamos recomeçar a busca nas notas para cada estudante?**


## Dois ponteiros

```text
101 Ana             101 9.0
 ↑                   ↑
 i                   j
```

- chaves iguais → combina e avança;
- chave da esquerda menor → avança a esquerda;
- chave da direita menor → avança a direita.

**Nenhum ponteiro precisa voltar.**


## Sort-merge: implementação {.smaller}

::: {.notes}
Percorrer devagar os primeiros passos. Os comentários indicam as três decisões possíveis. O objetivo é compreender por que o algoritmo descarta elementos sem voltar.
:::


In [ ]:
def combinar_ordenado(estudantes_ordenados, notas_ordenadas, mostrar_passos=False):
    i = 0
    j = 0
    resultado = []

    # Os ponteiros só avançam: nenhum elemento já descartado precisa ser revisto.
    while i < len(estudantes_ordenados) and j < len(notas_ordenadas):
        estudante = estudantes_ordenados[i]
        registro_nota = notas_ordenadas[j]

        mat_estudante = estudante["matricula"]
        mat_nota = registro_nota["matricula"]

        if mostrar_passos:
            print(f"i={i} ({mat_estudante}) | j={j} ({mat_nota})")

        if mat_estudante == mat_nota:
            # Encontramos a correspondência atual.
            resultado.append({
                "matricula": mat_estudante,
                "nome": estudante["nome"],
                "nota": registro_nota["nota"],
            })
            i += 1
            j += 1

        elif mat_estudante < mat_nota:
            # A nota atual já tem matrícula maior.
            # Logo, este estudante não aparecerá mais adiante nas notas.
            i += 1

        else:
            # A matrícula da nota é menor que a do estudante atual.
            # Logo, esta nota não encontrará correspondência mais adiante.
            j += 1

    return resultado


### Uma observação sobre `lambda`

Neste código, `lambda` cria uma função pequena usada apenas para informar ao `sorted` **qual valor deve ser usado na ordenação**:

```python
lambda x: x["matricula"]
```

Aqui ela equivale a:

```python
def obter_matricula(x):
    return x["matricula"]
```

Assim, `sorted(estudantes, key=lambda x: x["matricula"])` significa: **ordene os estudantes usando a matrícula como chave**.

Não é necessário dominar `lambda` nesta disciplina; ela aparece apenas como uma forma compacta de fornecer a função de ordenação.


In [ ]:
estudantes_ord = sorted(estudantes, key=lambda x: x["matricula"])
notas_ord = sorted(notas, key=lambda x: x["matricula"])

resultado_ordenado = combinar_ordenado(
    estudantes_ord,
    notas_ord,
    mostrar_passos=True
)

print("\nResultado:", resultado_ordenado)


## O percurso nunca volta

```text
101 == 101 → combina; avança ambos
102 <  103 → avança estudantes
103 == 103 → combina; avança ambos
104 == 104 → combina
```

Se as coleções **já estão ordenadas**:

$$O(n+m)$$


## Mas quem pagou pela ordenação?

Se precisamos ordenar antes:

$$O(n\log n) + O(m\log m) + O(n+m)$$

Se a ordenação já existe ou é reutilizada:

$$O(n+m)$$

**O estado inicial dos dados faz parte da análise do algoritmo.**


## Três estratégias, um problema

| Estratégia | Ideia | Custo aproximado |
|---|---|---|
| Nested loop | comparar pares | $O(nm)$ |
| Hash join | construir índice hash | $O(n+m)$ médio |
| Sort-merge | ordenar + percorrer | $O(n\log n + m\log m)$ |
| Sort-merge já ordenado | dois ponteiros | $O(n+m)$ |

**A estrutura disponível altera o algoritmo que podemos usar.**


## Qual é melhor? {.question-slide}

O que você escolheria se...

- as duas coleções já estivessem ordenadas?
- uma coleção fosse muito pequena?
- o mesmo índice fosse reutilizado muitas vezes?
- houvesse pouca memória disponível?
- uma estrutura hash já existisse?

**Big-O informa parte da decisão; não substitui o contexto.**


## O que estamos fazendo?

Estamos relacionando registros de duas coleções por uma chave.

### JOIN

Até aqui, nossa regra foi:

> produzir resultado apenas quando encontramos correspondência.

Mas existe outra decisão: **o que fazer com quem não encontra par?**


## E Bruno? {.question-slide}

Bruno existe em `estudantes`, mas não em `notas`.

Ele deve:

- desaparecer?
- permanecer com nota ausente?

Essa escolha define a **semântica** da combinação.


## Inner join

Mantém apenas correspondências.

```text
101 Ana   9.0
103 Carla 8.5
104 Diego 7.5
```

Bruno não aparece porque não há matrícula correspondente em `notas`.


## Left join

Preserva todos os registros da estrutura à esquerda.

```text
101 Ana    9.0
102 Bruno  ausente
103 Carla  8.5
104 Diego  7.5
```

Agora ausência de correspondência também é informação.


## Outer e right

**Right join:** preserva todos os registros da direita.

**Outer join:** preserva registros dos dois lados, tenham ou não correspondência.

A pergunta correta não é “qual join é melhor?”, mas:

> **qual resultado representa o problema que queremos analisar?**


## Agora sim: `pandas.merge()`

Uma chamada curta expressa a semântica desejada:

```python
df_estudantes.merge(df_notas, on="matricula", how="left")
```

Mas agora sabemos que “combinar” envolve **chaves, busca, organização, correspondências e custo**.


In [ ]:
import pandas as pd

df_estudantes = pd.DataFrame(estudantes)
df_notas = pd.DataFrame(notas)

resultado_left = df_estudantes.merge(
    df_notas,
    on="matricula",
    how="left",
)

resultado_left


## Uma nova pergunta: a chave é única?

Até agora, cada matrícula apareceu no máximo uma vez em cada estrutura.

Mas considere:

```text
101 → Estruturas de Dados
101 → Cálculo
101 → Estatística
```

Uma chave pode participar de **várias correspondências**.


## Cardinalidade

| Relação | Exemplo |
|---|---|
| 1:1 | estudante ↔ cadastro único |
| 1:N | pedido ↔ itens do pedido |
| N:1 | vários itens ↔ um produto |
| N:N | estudantes ↔ disciplinas |

A cardinalidade ajuda a prever **quantas correspondências podem ser produzidas**.


## Quando os dois lados repetem a chave

```text
A          B
chave      chave
1          1
1          1
```

Cada ocorrência de A encontra cada ocorrência de B:

$$2 \times 2 = 4$$

**O resultado pode ter mais linhas do que qualquer entrada.**


In [ ]:
a = pd.DataFrame({"chave": [1, 1], "origem_a": ["A1", "A2"]})
b = pd.DataFrame({"chave": [1, 1], "origem_b": ["B1", "B2"]})

a.merge(b, on="chave", how="inner")


## N:N pode multiplicar registros

Se uma chave aparece:

- `a` vezes à esquerda;
- `b` vezes à direita;

essa chave pode produzir:

$$a \times b$$

linhas no resultado.

Isso não é necessariamente erro. O problema é **não saber se essa multiplicação era esperada**.


## Antes do merge: quatro perguntas {.question-slide}

Não execute o `merge` imediatamente.

1. **Qual é a chave?**  
   O que ela representa no domínio?

2. **Ela deveria ser única em qual lado?**  
   A regra do problema prevê 1:1, 1:N, N:1 ou N:N?

3. **Ela realmente é única nos dados?**  
   Regra esperada e dado observado podem divergir.

4. **Quantas linhas esperamos depois?**  
   O resultado deve preservar, reduzir ou multiplicar registros?

::: {.notes}
Este é um procedimento de diagnóstico, não uma lista para decorar. Construir um exemplo: orders deveria ter order_id único; order_items não. Logo orders→order_items é 1:N. Perguntar o que significaria encontrar order_id duplicado em orders.
:::


## Diagnóstico 1 — tamanho e unicidade {.smaller}

Antes de combinar, investigamos as estruturas.


In [ ]:
def diagnosticar_chave(df, chave, nome):
    total = len(df)
    chaves_distintas = df[chave].nunique(dropna=False)
    duplicados = df.duplicated(subset=[chave], keep=False).sum()

    print(f"{nome}")
    print(f"  linhas: {total}")
    print(f"  chaves distintas: {chaves_distintas}")
    print(f"  chave única? {df[chave].is_unique}")
    print(f"  linhas envolvidas em duplicatas: {duplicados}")

diagnosticar_chave(df_estudantes, "matricula", "estudantes")
diagnosticar_chave(df_notas, "matricula", "notas")


## Diagnóstico 2 — onde estão as duplicatas?

Saber que “há duplicatas” ainda é pouco.

Queremos descobrir **quais chaves repetem e quantas vezes**.


In [ ]:
def frequencias_repetidas(df, chave):
    frequencias = df[chave].value_counts(dropna=False)

    # Apenas chaves com mais de uma ocorrência podem multiplicar correspondências.
    return frequencias[frequencias > 1]

print("Repetições em estudantes:")
print(frequencias_repetidas(df_estudantes, "matricula"))

print("\nRepetições em notas:")
print(frequencias_repetidas(df_notas, "matricula"))


## Diagnóstico 3 — declare a cardinalidade esperada

O Pandas pode verificar algumas expectativas durante o `merge`:

```python
validate="one_to_one"
validate="one_to_many"
validate="many_to_one"
validate="many_to_many"
```

Se os dados violarem a expectativa, queremos descobrir **antes de interpretar o resultado**.


In [ ]:
# Neste exemplo, esperamos no máximo uma linha por matrícula nos dois lados.
resultado_validado = df_estudantes.merge(
    df_notas,
    on="matricula",
    how="left",
    validate="one_to_one",
)

resultado_validado


## Diagnóstico 4 — compare antes e depois

Após o merge, não olhe apenas as primeiras linhas.

```text
linhas antes
cardinalidade esperada
        ↓
      MERGE
        ↓
linhas depois
chaves repetidas
ausências
```

**A quantidade de linhas é uma evidência sobre o que aconteceu.**


In [ ]:
antes_esquerda = len(df_estudantes)
antes_direita = len(df_notas)
depois = len(resultado_validado)

print("Linhas em estudantes:", antes_esquerda)
print("Linhas em notas:", antes_direita)
print("Linhas após o left join:", depois)
print("Variação em relação à esquerda:", depois - antes_esquerda)


## AGORA É COM VOCÊ {.question-slide}

**Notebook Estudante · Atividade 3 — Antes de confiar no merge**

Para duas pequenas tabelas fornecidas:

1. identifique a chave;
2. formule a cardinalidade esperada;
3. verifique unicidade e duplicatas;
4. preveja se o número de linhas vai diminuir, permanecer ou aumentar;
5. execute o merge com `validate`;
6. explique qualquer diferença entre previsão e resultado.

[**Abrir Notebook Estudante no Colab**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana09/EGC5310-EstruturasDados-S09-04-Estudante.ipynb)


## Do exemplo pequeno para o Olist

No Olist, as informações estão distribuídas em tabelas relacionadas:

```text
customers
    │
    ▼
orders
    │
    ▼
order_items
    │
    ▼
products
```

Agora a cardinalidade deixa de ser um exemplo artificial.


## Um pedido tem quantos itens? {.question-slide}

Antes de combinar `orders` e `order_items`:

- `order_id` deveria ser único em `orders`?
- deveria ser único em `order_items`?
- esperamos 1:1, 1:N, N:1 ou N:N?
- o resultado deve ter o mesmo número de linhas de `orders`?

**Registre a hipótese antes de executar.**


## Carregando Olist

O Notebook Mestre pressupõe os CSVs oficiais do conjunto Olist disponíveis em `dados/olist/`.

A atividade deve continuar funcionando no Colab depois que os arquivos forem obtidos pelo procedimento definido para a semana.

::: {.notes}
Na preparação final da semana, alinhar este caminho com o mecanismo de download escolhido no Notebook Estudante/Professor. Não esconder a origem dos dados em código de infraestrutura.
:::


### Pré-requisito para o bloco Olist

Este notebook espera:

```text
dados/olist/olist_orders_dataset.csv
dados/olist/olist_order_items_dataset.csv
```

O código verifica a existência dos arquivos antes da leitura, evitando interromper o restante do notebook.

> **Antes da aula:** disponibilizar esses CSVs no repositório/ambiente da disciplina ou orientar a turma sobre o procedimento de obtenção adotado para a Semana 09. O notebook não faz download automático.


In [ ]:
from pathlib import Path

pasta_olist = Path("dados/olist")

arquivo_orders = pasta_olist / "olist_orders_dataset.csv"
arquivo_items = pasta_olist / "olist_order_items_dataset.csv"

if arquivo_orders.exists() and arquivo_items.exists():
    orders = pd.read_csv(arquivo_orders)
    order_items = pd.read_csv(arquivo_items)

    print("orders:", orders.shape)
    print("order_items:", order_items.shape)
else:
    print("Arquivos Olist ainda não disponíveis neste ambiente.")


## Antes do merge real: diagnostique

Quando os arquivos estiverem disponíveis, a sequência é:

```python
diagnosticar_chave(orders, "order_id", "orders")
diagnosticar_chave(order_items, "order_id", "order_items")
```

**Primeiro observamos as propriedades das chaves. Depois combinamos.**


In [ ]:
if arquivo_orders.exists() and arquivo_items.exists():
    diagnosticar_chave(orders, "order_id", "orders")
    diagnosticar_chave(order_items, "order_id", "order_items")


## Execute com uma expectativa explícita

Se `orders.order_id` é único e `order_items.order_id` pode repetir:

```text
orders 1 ───── N order_items
```

Então, do ponto de vista de `orders` → `order_items`, esperamos **one-to-many**.


In [ ]:
if arquivo_orders.exists() and arquivo_items.exists():
    pedidos_itens = orders.merge(
        order_items,
        on="order_id",
        how="inner",
        validate="one_to_many",
    )

    print("orders:", len(orders))
    print("order_items:", len(order_items))
    print("pedidos_itens:", len(pedidos_itens))


## Por que o número de linhas mudou?

Um pedido pode gerar várias linhas após a combinação:

```text
pedido 001
   ├── item 1
   ├── item 2
   └── item 3
```

O `merge` não “duplicou pedidos por engano”.

**Ele materializou a relação 1:N.**


## Construindo uma visão analítica

Ao acrescentar `products` e `customers`, precisamos repetir o raciocínio:

1. qual é a chave?
2. em qual lado ela é única?
3. qual cardinalidade esperamos?
4. qual crescimento de linhas faz sentido?

**Cada merge é uma hipótese sobre a estrutura dos dados.**


## Benchmark da semana {.question-slide}

Vamos comparar três estratégias para o mesmo problema:

- nested-loop;
- hash join;
- sort-merge.

Antes de medir:

> Qual curva você espera para cada estratégia quando `n` cresce?


## Experimento controlado

Tamanhos iniciais:

```text
500
1.000
2.500
5.000
10.000
```

O limite final será calibrado porque o nested-loop pode rapidamente dominar o tempo de execução.

**Mesmos dados lógicos; estratégias diferentes.**


## O que o benchmark deve separar?

Não queremos apenas “quem ganhou”.

Precisamos distinguir:

- custo de construir o hash;
- custo de consultar o hash;
- custo de ordenar;
- custo do percurso sort-merge;
- situação em que os dados já estavam ordenados;
- possibilidade de reutilizar uma estrutura preparada.

**Preparação também é computação.**


## Não existe uma estratégia universal

| Situação | Questão relevante |
|---|---|
| dados pequenos | simplicidade pode bastar |
| hash já disponível | consulta pode ser muito vantajosa |
| dados já ordenados | sort-merge evita nova ordenação |
| índice reutilizado | custo inicial é amortizado |
| pouca memória | estrutura auxiliar importa |

O algoritmo deve ser analisado **junto com o estado e a representação dos dados**.


## O caminho da semana

```text
duas coleções
      ↓
nested-loop
      ↓
custo da busca repetida
      ↓
hashing ───── ordenação
      ↓           ↓
 hash join    sort-merge
       \         /
          JOIN
           ↓
       semântica
           ↓
     cardinalidade
           ↓
   diagnóstico + Pandas
           ↓
          Olist
```


## Uma linha, muitas decisões {.question-slide}

```python
df1.merge(df2, on="id")
```

Agora conseguimos perguntar:

- como as chaves serão encontradas?
- que organização dos dados ajuda?
- quais correspondências serão preservadas?
- a chave é única?
- quantas linhas podem surgir?
- o resultado corresponde à cardinalidade esperada?

**A API ficou curta. O problema computacional não.**


## Ponte para a Semana 10

Depois dos joins, podemos chegar a uma visão como:

```text
cliente | pedido | produto | categoria | preço
```

Agora surge outra pergunta:

> **Como resumir milhões de registros para responder quanto cada categoria vendeu?**

### Próxima semana: agrupamentos e agregações
